In [ ]:
!nvidia-smi

In [ ]:
!pip install --no-cache-dir \
  "torch==2.10.0" \
  "transformers==4.57.1" \
  "datasets==3.6.0" \
  "accelerate==1.12.0" \
  "peft==0.18.0" \
  "trl==0.26.0" \
  "bitsandbytes==0.49.0" \
  "huggingface_hub==0.36.0" \
  "wandb==0.23.0" \
  "fsspec==2025.3.0"

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb
import accelerate

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

In [ ]:
# START THE TRAINING

In [ ]:
import os
import torch
import gc

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

login(token=hf_token)

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

base_model_id = "Qwen/Qwen2-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

model.config.use_cache = False

print("Model loaded in FP16 LoRA mode")

Model loaded in FP16 LoRA mode


In [4]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

In [6]:
from datasets import load_dataset

dataset_repo = "yowww1094/tourism-llm-fine-tuning-dataset"

dataset = load_dataset(dataset_repo, split="train")

split_dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Eval size:", len(eval_dataset))

Train size: 130
Eval size: 15


In [7]:
print(train_dataset[0])

{'messages': [{'role': 'system', 'content': 'You are a tourism assistant. Answer using the provided context. If the context is based on social media or traveler opinions, express uncertainty and avoid presenting it as official fact.'}, {'role': 'user', 'content': "Context:\n[1] Things to do in Agadir I will be travelling to Agadir next week. Anything interesting I should do?\n\n[2] Going to Agadir - advice please Hello,\\n\\nI am heading to Agadir soon, staying in hotel riu tikda dunas.\\n\\nPlanning a few trips, wanted some advice about safest way to travel and whether the below are possible as day trips\\n\\nSpecific areas: \\n\\nSouk el had d'Agadir, \\n\\nParadise valley,\\n\\nLa Medina Agadir,\\n\\nHasan 2 mosque, Casablanca,\\n\\nKasbah Agadir Oufella,\\n\\nWintimdouine,\\n\\nInzerki,\\n\\n\n\n[3] Oh no, Agadir is a really safe city, with TONS AND TONS AND TONS of tourists. People come here and never leave, and you actually chose one of the best moments to come to Agadir,its the 

In [8]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen2-tourism-assistant-chatbot-lora",

    max_length=512,
    packing=False,

    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,

    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    logging_steps=5,
    save_steps=20,
    eval_strategy="steps",
    eval_steps=20,
    save_total_limit=2,

    fp16=True,
    bf16=False,

    optim="adamw_torch",   # no bitsandbytes
    gradient_checkpointing=True,

    report_to="none",
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,2.671000,2.902138,2.718750,149119.000000,0.448019


TrainOutput(global_step=27, training_loss=2.7242767016092935, metrics={'train_runtime': 3985.3567, 'train_samples_per_second': 0.098, 'train_steps_per_second': 0.007, 'total_flos': 1571198226923520.0, 'train_loss': 2.7242767016092935, 'entropy': 2.6851128472222223, 'num_tokens': 199113.0, 'mean_token_accuracy': 0.4532845947477553, 'epoch': 3.0})

In [10]:
eval_results = trainer.evaluate()
print(eval_results)

{'eval_loss': 2.8923377990722656, 'eval_entropy': 2.716552734375, 'eval_num_tokens': 214402.0, 'eval_mean_token_accuracy': 0.45022016018629074}


In [11]:
import math

eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print("Eval loss:", eval_loss)
print("Perplexity:", perplexity)

Eval loss: 2.8923377990722656
Perplexity: 18.035423551840303


In [12]:
output_dir = "/kaggle/working/qwen2-ai-travel-buddy-lora"

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Saved LoRA adapter to:", output_dir)

Saved LoRA adapter to: /kaggle/working/qwen2-ai-travel-buddy-lora


In [13]:
import gc
import torch

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_id = "Qwen/Qwen2-1.5B-Instruct"
adapter_dir = "/kaggle/working/qwen2-ai-travel-buddy-lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

In [15]:
merged_model = PeftModel.from_pretrained(
    base_model,
    adapter_dir
)

In [16]:
merged_model = merged_model.merge_and_unload()

print("LoRA adapter merged into base model.")

LoRA adapter merged into base model.


In [17]:
merged_output_dir = "/kaggle/working/qwen2-tourism-assistant-chatbot-merged"

merged_model.save_pretrained(
    merged_output_dir,
    safe_serialization=True
)

tokenizer.save_pretrained(merged_output_dir)

print("Merged model saved to:", merged_output_dir)

Merged model saved to: /kaggle/working/qwen2-ai-travel-buddy-merged


In [18]:
!ls -lh /kaggle/working/qwen2-tourism-assistant-chatbot-merged

total 2.9G
-rw-r--r-- 1 root root   80 Apr 29 12:12 added_tokens.json
-rw-r--r-- 1 root root  328 Apr 29 12:12 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Apr 29 12:12 config.json
-rw-r--r-- 1 root root  242 Apr 29 12:12 generation_config.json
-rw-r--r-- 1 root root 1.6M Apr 29 12:12 merges.txt
-rw-r--r-- 1 root root 2.9G Apr 29 12:12 model.safetensors
-rw-r--r-- 1 root root  367 Apr 29 12:12 special_tokens_map.json
-rw-r--r-- 1 root root  973 Apr 29 12:12 tokenizer_config.json
-rw-r--r-- 1 root root  11M Apr 29 12:12 tokenizer.json
-rw-r--r-- 1 root root 2.7M Apr 29 12:12 vocab.json


In [19]:
from huggingface_hub import create_repo

hf_model_repo = "yowww1094/tourism-llm-fine-tuned-qwen2-1.5b-lora-merged"

create_repo(
    repo_id=hf_model_repo,
    repo_type="model",
    private=True,
    exist_ok=True
)

merged_model.push_to_hub(
    hf_model_repo,
    private=True,
    safe_serialization=True
)

tokenizer.push_to_hub(
    hf_model_repo,
    private=True
)

print("Merged model pushed to:", hf_model_repo)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Merged model pushed to: yowww1094/tourism-llm-fine-tuned-qwen2-1.5b-lora-merged


In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

hf_model_repo = "yowww1094/tourism-llm-fine-tuned-qwen2-1.5b-lora-merged"

tokenizer = AutoTokenizer.from_pretrained(hf_model_repo)

model = AutoModelForCausalLM.from_pretrained(
    hf_model_repo,
    dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

model.eval()
print("Merged model loaded for inference.")

tokenizer_config.json:   0%|          | 0.00/973 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/328 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Merged model loaded for inference.


In [24]:
messages = [
    {
        "role": "system",
        "content": "You are AI Travel Buddy, a helpful Moroccan tourism assistant. Answer clearly and safely in the user's language."
    },
    {
        "role": "user",
        "content": "give me 2 days plan in agadir"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

system
You are AI Travel Buddy, a helpful Moroccan tourism assistant. Answer clearly and safely in the user's language.
user
give me 2 days plan in agadir
assistant
Sure! Here is a two-day itinerary for Agadir:

Day 1:
8:00 AM - Start your day with a visit to the Souk el Had, Agadir's bustling market filled with traditional Moroccan textiles, spices, souvenirs, and local crafts.
11:30 AM - Head over to the Jardin Majorelle, which was once owned by French artist Jacques Majorelle and now houses an incredible collection of plants and flowers from around the world.
1:00 PM - Continue your exploration of the city by visiting the El Badi Kasbah, a UNESCO World Heritage Site that offers breathtaking views of the city and a glimpse into Morocco's rich past.
5:00 PM - End your first day at the beach or explore the city further.

Day 2:
8:00 AM - Take a scenic drive through the countryside to visit the nearby desert oasis town of Essaouira, known for its vibrant colors and friendly locals.
11:0